In [ ]:
# Purpose: imports and small setup
import logging
logger = logging.getLogger(__name__)
if not logger.handlers:
    logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')


In [ ]:
# Purpose: plotting / figure generation
# Reproducibility helper added by tools/add_repro_cell.py
import os, sys, random, json
from datetime import datetime
import numpy as np
seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)
np.random.seed(seed)
# Optional: set torch deterministic flags if torch is available
try:
    import torch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # deterministic settings (may affect performance)
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
except Exception:
    pass
# Record environment info for reproducibility
info = dict(
    python=sys.executable,
    python_version='.'.join(map(str, sys.version_info[:3])),
    numpy=np.__version__,
    seed=seed,
    timestamp=str(datetime.utcnow())
)
# Add optional package checks
try:
    import pkg_resources
    pkgs=['nbformat','pandas','scipy','sklearn','matplotlib','seaborn']
    pv = {p: pkg_resources.get_distribution(p).version for p in pkgs if pkg_resources.get_distribution(p) is not None}
    info['packages'] = pv
except Exception:
    pass
logger.info('REPRO: ' + json.dumps(info))


In [7]:
# Purpose: imports and small setup
from google.colab import drive, userdata
from pathlib import Path

# (commented) drive.mount("/content/drive")

# (sanitized path) PROJECT_ROOT = Path("PROJECT_ROOT")
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project root missing: {PROJECT_ROOT}")

REQUIRED_DIRS = ["prompts", "configs", "scripts", "python_domain", "clinical_domain", "history_domain", "tests"]
missing = [d for d in REQUIRED_DIRS if not (PROJECT_ROOT / d).exists()]
if missing:
    raise FileNotFoundError(f"Missing repo subdirectories: {missing}")

logger.info(f"Project root: {PROJECT_ROOT}")
logger.info(f"Subdirectories present: {REQUIRED_DIRS}")


Mounted at /content/drive
Project root: /content/drive/MyDrive/Dr Dacon ICDM/SeamSFF-main
Subdirectories present: ['prompts', 'configs', 'scripts', 'python_domain', 'clinical_domain', 'history_domain', 'tests']


In [8]:
# Purpose: data processing / analysis step
ANTHROPIC_KEY = userdata.get("Anthropic")
OPENAI_KEY = userdata.get("OPENAI_API_KEY")
GROQ_KEY = userdata.get("Groq")

for name, value in [("Anthropic", ANTHROPIC_KEY), ("OpenAI", OPENAI_KEY), ("Groq", GROQ_KEY)]:
    if not value:
        raise RuntimeError(f"Secret not loaded: {name}")
logger.info("Secrets loaded.")


Secrets loaded.


In [9]:
# Purpose: data processing / analysis step
!pip install -q anthropic openai groq


In [10]:
# Purpose: imports and small setup
import anthropic
from openai import OpenAI
from groq import Groq

PROMPTS_DIR = PROJECT_ROOT / "prompts"

MODELS = {
    "anthropic": "claude-haiku-4-5",
    "openai": "gpt-3.5-turbo",
    "groq": "llama-3.3-70b-versatile",
}

PROMPT_FILES = {
    "seamless_agent": "seamless_control_agent.md",
    "sff_agent": "sff_treatment_agent.md",
    "persona_novice": "persona_novice.md",
    "persona_intermediate": "persona_intermediate.md",
    "persona_expert": "persona_expert.md",
    "judge": "judge_quality_scorer.md",
    "state_tagger": "state_tagger.md",
}

prompts = {}
for name, filename in PROMPT_FILES.items():
    path = PROMPTS_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Prompt missing: {path}")
    prompts[name] = path.read_text(encoding="utf-8")
    logger.info(f"{name:24s}  {len(prompts[name]):6d} chars  {filename}")


seamless_agent              3615 chars  seamless_control_agent.md
sff_agent                   8052 chars  sff_treatment_agent.md
persona_novice              3009 chars  persona_novice.md
persona_intermediate        3119 chars  persona_intermediate.md
persona_expert              3118 chars  persona_expert.md
judge                       4589 chars  judge_quality_scorer.md
state_tagger                2475 chars  state_tagger.md


In [12]:
# Purpose: imports and small setup
import json

CONFIG_PATH = PROJECT_ROOT / "configs" / "project_config.json"
with CONFIG_PATH.open() as f:
    project_config = json.load(f)

shape = project_config["study_shape"]
sampling = project_config["sampling_protocol"]
scoring = project_config["scoring_protocol"]

logger.info(f"Project: {project_config['project']}")
logger.info(f"Domains: {project_config['domains']}")
logger.info(f"Benchmarks per domain:")
for domain, benchmarks in project_config["benchmarks"].items():
    logger.info(f"  {domain}: {benchmarks}")
logger.info(f"Tasks per benchmark: {shape['tasks_per_benchmark']} (planned base: {shape['planned_base_task_count']})")
logger.info(f"Difficulty targets: {shape['difficulty_targets']}")
logger.info(f"Judge role: {scoring['judge_role']}")
logger.info(f"Normalized quality scale: {scoring['normalization_policy']['normalized_quality_scale']}")
logger.info(f"Calibration sample per domain: {scoring['calibration_sample_per_domain']}, min Pearson r: {scoring['minimum_pearson_r']}")


Project: ICDM 2026 SFF simulation-and-data-mining project
Domains: ['python_debugging', 'clinical_text_interpretation', 'historical_document_synthesis']
Benchmarks per domain:
  python_debugging: ['MBPP', 'HumanEval', 'APPS']
  clinical_text_interpretation: ['MedQA', 'MedMCQA', 'PubMedQA', 'MMLU medical subset']
  historical_document_synthesis: ['World-History-1500 QA', 'ArchivalQA', 'ChroniclingAmericaQA']
Tasks per benchmark: 60 (planned base: 600)
Difficulty targets: {'easy': 20, 'medium': 20, 'hard': 20}
Judge role: blinded_deterministic_technical_auditor
Normalized quality scale: 0-4
Calibration sample per domain: 20, min Pearson r: 0.75
